In [0]:
-- 1. Retrieve all records from the employee table.
SELECT * FROM ska_catalog.bronze.employee;

In [0]:
-- 2. Retrieve all records from the department table.
SELECT * FROM ska_catalog.bronze.department;

In [0]:
-- 3. Select the first name and last name of all employees.
SELECT FIRST_NAME, LAST_NAME FROM ska_catalog.bronze.employee;

In [0]:
-- 4. Select the department name and location ID from the department table.
SELECT DEPARTMENT_NAME, LOCATION_ID FROM ska_catalog.bronze.department;

In [0]:
-- 5. Find the total number of employees.
SELECT count(*) AS total_employees FROM ska_catalog.bronze.employee;


In [0]:
-- 6. Find the total number of departments.
SELECT count(*) AS total_dept FROM ska_catalog.bronze.department;

In [0]:
-- 7. Retrieve employees with salary greater than 5000. and less tha 10000
SELECT CONCAT(a.FIRST_NAME , ' ' , a.LAST_NAME) AS `FULL NAME`, a.SALARY FROM ska_catalog.bronze.employee a
WHERE a.SALARY BETWEEN 5000 AND 10000
ORDER BY a.SALARY DESC;

In [0]:
-- ALTER TABLE ska_catalog.bronze.employee
-- ADD COLUMN FLAG CHAR(1);

In [0]:
-- INSERT INTO TABLE ska_catalog.bronze.employee
-- VALUES (198,	'Donald'	,'OConnell','DOCONNEL','650.507.9833','21-JUN-07','SH_CLERK',2600,'-' ,	'124','50',null)

In [0]:
UPDATE ska_catalog.bronze.employee
SET FLAG = 'Y'
WHERE LAST_NAME IN (
  SELECT  LAST_NAME FROM ska_catalog.bronze.employee
  GROUP BY LAST_NAME
  HAVING count(*) > 1
);
SELECT * FROM ska_catalog.bronze.employee;

In [0]:
-- DELETING duplicate rows
DELETE FROM ska_catalog.bronze.employee
WHERE EMPLOYEE_ID IN (
    SELECT EMPLOYEE_ID
    FROM (
        SELECT EMPLOYEE_ID, ROW_NUMBER() OVER (PARTITION BY EMPLOYEE_ID ORDER BY EMPLOYEE_ID) AS rn
        FROM ska_catalog.bronze.employee
    ) sub
    WHERE rn > 1
);

In [0]:
-- 8. Retrieve departments located in location ID 1700.

SELECT DEPARTMENT_ID, DEPARTMENT_NAME FROM 
ska_catalog.bronze.department
WHERE LOCATION_ID = 1700

In [0]:
-- 1. Retrieve employee details along with their department name using JOIN.

SELECT a.FIRST_NAME, b.DEPARTMENT_NAME FROM ska_catalog.bronze.employee a
LEFT JOIN ska_catalog.bronze.department b
ON a.DEPARTMENT_ID = b.DEPARTMENT_ID

In [0]:
-- 2. Find the average salary for each department.
SELECT  dept.DEPARTMENT_ID,
        dept.DEPARTMENT_NAME,
        COALESCE(ROUND(AVG(emp.salary),4), 0) AS `Average_salary`
FROM ska_catalog.bronze.department dept
LEFT JOIN ska_catalog.bronze.employee emp
ON emp.DEPARTMENT_ID = dept.DEPARTMENT_ID
GROUP BY dept.DEPARTMENT_ID, dept.DEPARTMENT_NAME
ORDER BY Average_salary DESC;

In [0]:
-- Retrieve employees who do not have a manager.
UPDATE ska_catalog.bronze.employee
SET MANAGER_ID = NULL
WHERE MANAGER_ID = ' - ';

In [0]:
SELECT * FROM ska_catalog.bronze.employee
WHERE MANAGER_ID IS NULL;

In [0]:
ALTER TABLE ska_catalog.bronze.employee
SET TBLPROPERTIES (
  'delta.columnMapping.mode' = 'name'
);

In [0]:
-- ALTER TABLE ska_catalog.bronze.employee
-- DROP COLUMN COMMISSION_PCT;

In [0]:
SHOW TBLPROPERTIES ska_catalog.bronze.employee

In [0]:
SELECT deprt.DEPARTMENT_ID,deprt.DEPARTMENT_NAME, ROUND(AVG(emp.SALARY ),2) AS `AVG SALARY` FROM ska_catalog.bronze.employee emp
INNER JOIN ska_catalog.bronze.department deprt
ON emp.DEPARTMENT_ID = deprt.DEPARTMENT_ID
GROUP BY deprt.DEPARTMENT_ID,deprt.DEPARTMENT_NAME
ORDER BY `AVG SALARY` DESC;

In [0]:
%sql
-- UPDATE ska_catalog.bronze.employee 
-- SET MANAGER_ID = NULL
-- WHERE MANAGER_ID = ' - '

In [0]:
%sql
SELECT 
  emp1.FIRST_NAME || ' ' || emp1.LAST_NAME AS employee_name,
  emp2.FIRST_NAME || ' ' || emp2.LAST_NAME AS manager_name
FROM ska_catalog.bronze.employee emp1
JOIN ska_catalog.bronze.employee emp2
  ON emp1.MANAGER_ID = emp2.EMPLOYEE_ID

In [0]:
%sql
-- Retrive all the MANAGER and COUNT of employees under them.
SELECT CONCAT(emp2.FIRST_NAME, " ",emp2.LAST_NAME) AS `MANAGER`,COUNT(emp1.EMPLOYEE_ID) AS  `EMP COUNT` 
FROM ska_catalog.bronze.employee emp1
JOIN ska_catalog.bronze.employee emp2
ON emp1.MANAGER_ID = emp2.EMPLOYEE_ID
GROUP BY `MANAGER`
ORDER BY  `EMP COUNT` DESC;

In [0]:
-- SELECT TOP 5 MANAGER with highest no of employees under them and count greated than 5.
-- use HAVING.
SELECT CONCAT(emp2.FIRST_NAME, " ",emp2.LAST_NAME) AS `MANAGER`,COUNT(emp1.EMPLOYEE_ID) AS  `EMP COUNT` 
FROM ska_catalog.bronze.employee emp1
JOIN ska_catalog.bronze.employee emp2
ON emp1.MANAGER_ID = emp2.EMPLOYEE_ID
GROUP BY `MANAGER`
HAVING COUNT(emp1.EMPLOYEE_ID) >= 5
ORDER BY  `EMP COUNT` DESC
LIMIT 5;

In [0]:
SELECT  * FROM ska_catalog.bronze.employee;
SELECT  * FROM ska_catalog.bronze.department;

In [0]:
-- 10. Retrieve departments with employees having job ID 'ST_CLERK'.
SELECT DISTINCT deprt.DEPARTMENT_NAME
FROM ska_catalog.bronze.employee emp
INNER JOIN ska_catalog.bronze.department deprt
  ON emp.DEPARTMENT_ID = deprt.DEPARTMENT_ID
WHERE emp.JOB_ID = 'ST_CLERK'

In [0]:
SELECT
  emp1.FIRST_NAME || ' ' || emp1.LAST_NAME AS `EMPLOYEE NAME`
FROM ska_catalog.bronze.employee emp1
JOIN ska_catalog.bronze.employee emp2
  ON emp1.MANAGER_ID = emp2.EMPLOYEE_ID
WHERE  emp1.JOB_ID = emp2.JOB_ID

In [0]:
-- 4. Retrieve employees whose salary is more than their manager.
SELECT
  emp1.FIRST_NAME || ' ' || emp1.LAST_NAME AS employee_name,
  emp1.SALARY AS employee_salary,
  emp2.FIRST_NAME || ' ' || emp2.LAST_NAME AS manager_name,
  emp2.SALARY AS manager_salary
FROM ska_catalog.bronze.employee emp1
JOIN ska_catalog.bronze.employee emp2
  ON emp1.MANAGER_ID = emp2.EMPLOYEE_ID

In [0]:
%skip
-- 1. Add a new column with DATE type
ALTER TABLE ska_catalog.bronze.employee
ADD COLUMN HIRE_DATE_NEW DATE;
-- 2. Update the new column by parsing the old column with the correct format
UPDATE ska_catalog.bronze.employee
SET HIRE_DATE_NEW = to_date(HIRE_DATE, 'dd-MMM-yy');
-- 3. Drop the old column
ALTER TABLE ska_catalog.bronze.employee
DROP COLUMN HIRE_DATE;
-- 4. Rename the new column to the original name
ALTER TABLE ska_catalog.bronze.employee
RENAME COLUMN HIRE_DATE_NEW TO HIRE_DATE;

In [0]:
SELECT * FROM ska_catalog.bronze.employee
WHERE YEAR(HIRE_DATE) = 2005
ORDER BY EMPLOYEE_ID;

In [0]:
SELECT YEAR(HIRE_DATE) AS YEAR , COUNT(EMPLOYEE_ID) AS EMP_COUNT 
FROM ska_catalog.bronze.employee
GROUP BY YEAR
ORDER BY YEAR;